<!-- SPDX-FileCopyrightText: Copyright (c) 2026 DataRobot, Inc. and its affiliates. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# Mean-CVaR Optimization with JointFM Scenarios

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📘 About This Notebook</h4>
This notebook replaces the KDE scenario generator of <a href="cvar_basic.ipynb"><code>cvar_basic.ipynb</code></a> with <strong>JointFM</strong>, a foundation model that forecasts the joint distribution of several time series. Each scenario is one joint draw of the next day's asset returns, forecast from the look-back window of historical returns, so it stands for the same quantity as a KDE or Gaussian scenario and the optimizer's CVaR keeps its meaning.
<br><br>
The optimizer never talks to the model directly: <code>fit_type="jointfm"</code> calls the <a href="https://github.com/datarobot/joint-client-python"><code>jointfm-client</code></a> SDK, which reaches a JointFM service over HTTP. The same code runs against a JointFM instance on your machine, one DataRobot deployment, or a pool of deployments; only the <code>.env</code> file changes.
</div>

**Prerequisites**

- The `jointfm` extra: `uv sync --extra jointfm` (add your CUDA extra and `--group notebooks` as in the main README).
- A reachable JointFM service and a `.env` file in the repository root that selects it (section 1).
- The notebook solves on CPU with CLARABEL, so it needs no GPU.

## 1. Choose Where JointFM Runs

The client reads its connection settings from the `.env` file in the repository root, which `JointFMSettings(dotenv_path="../.env")` points at from this directory. Create it from the template and set exactly one selector; the rest of the notebook is the same for all four:

```bash
cp .env.sample .env  # in the repository root
```

| Selector | JointFM service |
| --- | --- |
| `JOINTFM_LOCAL_BASE_URL` | An instance you run yourself |
| `JOINTFM_LOCAL_BASE_URLS` | Several instances serving the same checkpoint, for example one per GPU |
| `JOINTFM_DEPLOYMENT_ID` | One DataRobot deployment (also needs `DATAROBOT_ENDPOINT` and `DATAROBOT_API_TOKEN`) |
| `JOINTFM_DEPLOYMENT_IDS` | A pool of DataRobot deployments serving the same checkpoint (same credentials); the client spreads each request's scenarios across them and retries on another one when a deployment fails |

`.env` is git-ignored, so the API token stays out of version control. A variable set in the process environment overrides the same variable in `.env`, but selectors from both places count together.

In [ ]:
import os

import cvxpy as cp
import pandas as pd
from jointfm_client import JointFMClient

from portfolio_optimization import cvar_optimizer, cvar_utils, utils
from portfolio_optimization.cvar_parameters import CvarParameters
from portfolio_optimization.settings import (
    ApiSettings,
    JointFMSettings,
    ReturnsComputeSettings,
    ScenarioGenerationSettings,
)

### 1.1 Check the Service

The health response says which model is served and how large a request it accepts. A scenario is a draw from the joint distribution of **all** assets, so the whole universe goes into one request and must fit within `max_series`; the simulator refuses a larger universe instead of splitting it. A longer look-back window than `n_input` rows is cut to its most recent `n_input` rows.

In [ ]:
jointfm_settings = JointFMSettings(dotenv_path="../.env")

client = JointFMClient.from_env(
    dotenv_path=jointfm_settings.dotenv_path,
    config_path=jointfm_settings.config_path,
)
health = client.health()
envelope = health.data_generation
instances = client.health_instances()

print(f"Model:            {health.model_version} ({health.generation})")
print(
    f"Instances:        {len(instances.instances)} ({instances.topology_label} samples)"
)
print(f"Instance IDs:     {[entry.instance_id for entry in instances.instances]}")
print(f"Assets/request:   {envelope.min_series} to {envelope.max_series}")
print(f"History rows:     up to {envelope.n_input}")

## 2. Data Preparation

<div style="background-color: #fff8e1; border-left: 6px solid #f9a825; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Disclaimer</h4>
Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
</div>

We take daily log returns of large-cap stocks from several sectors over `2021-01-01` to `2024-01-01`, keeping as many of them as the served model accepts in one request.

In [ ]:
data_path = "../data/stock_data/sp500.csv"
if not os.path.exists(data_path):
    utils.download_data(data_path)

candidate_tickers = [
    "NVDA",
    "MSFT",
    "XOM",
    "JPM",
    "UNH",
    "CAT",
    "AAPL",
    "PG",
    "JNJ",
    "AMZN",
]
tickers = candidate_tickers[: envelope.max_series]

prices = pd.read_csv(data_path, index_col=0, parse_dates=True)[tickers]
regime_dict = {"name": "recent", "range": ("2021-01-01", "2024-01-01")}
returns_compute_settings = ReturnsComputeSettings(return_type="LOG", freq=1)
returns_dict = utils.calculate_returns(prices, regime_dict, returns_compute_settings)

print(f"{len(tickers)} assets, {len(returns_dict['returns'])} daily returns")

## 3. Generate JointFM Scenarios

`fit_type="jointfm"` rebuilds each stock's log-price path from the look-back window's daily log returns, asks JointFM for `num_scen` joint draws of the next day's log prices, and converts each draw's difference to the last log price into the configured return type, so every scenario implies a positive price. The client splits a large `num_scen` across requests and pooled instances along the sample axis, so `num_scen` is not limited by one instance's capacity. Set `seed` on `ScenarioGenerationSettings` to make the draws reproducible.

`JointFMSettings.expected_return` chooses the expected returns the optimizer uses: `"scenario"` (the default), the mean of the JointFM scenarios, so that expected return and CVaR come from the same forecast, or `"historical"`, the look-back window's mean, as KDE uses.

In [ ]:
scenario_generation_settings = ScenarioGenerationSettings(
    num_scen=10000,
    fit_type="jointfm",
    jointfm_settings=jointfm_settings,
)
returns_dict = cvar_utils.generate_cvar_data(returns_dict, scenario_generation_settings)

scenarios = pd.DataFrame(returns_dict["cvar_data"].R.T, columns=tickers)
historical = returns_dict["returns"]
pd.DataFrame(
    {
        "historical mean": historical.mean(),
        "scenario mean": scenarios.mean(),
        "historical std": historical.std(),
        "scenario std": scenarios.std(),
        "scenario 5% quantile": scenarios.quantile(0.05),
    }
).round(4)

## 4. Solve the Mean-CVaR Problem

From here on the workflow is the same as for any other scenario generator: the scenarios sit in `returns_dict["cvar_data"]`, and the optimizer does not know where they came from. We build a long-only portfolio with at most 40% in one asset and up to 20% cash.

In [ ]:
cvar_params = CvarParameters(
    w_min=0.0,
    w_max=0.4,
    c_min=0.0,
    c_max=0.2,
    L_tar=1.0,
    risk_aversion=1,
    confidence=0.95,
)

cvar_problem = cvar_optimizer.CVaR(
    returns_dict=returns_dict,
    cvar_params=cvar_params,
    api_settings=ApiSettings(api="cvxpy"),
)
cpu_solver_settings = {
    "solver": cp.CLARABEL,
    "verbose": False,
    "tol_gap_abs": 1e-4,
    "tol_gap_rel": 1e-4,
    "tol_feas": 1e-4,
}
results, portfolio = cvar_problem.solve_optimization_problem(
    solver_settings=cpu_solver_settings
)

In [ ]:
results

## 5. Where to Go Next

- **Rebalancing.** `rebalance.rebalance_portfolio` accepts the same `ScenarioGenerationSettings`, so [`rebalancing_strategies.ipynb`](rebalancing_strategies.ipynb) runs on JointFM scenarios once its settings use `fit_type="jointfm"` and its universe fits `max_series`; each re-optimization forecasts from that date's look-back window.
- **Throughput.** For many scenarios or frequent re-optimization, point `.env` at a pool (`JOINTFM_DEPLOYMENT_IDS` or `JOINTFM_LOCAL_BASE_URLS`); the client spreads the scenarios across the instances.
- **Universe size.** The served model's `max_series` caps the universe; a larger universe needs a model trained for more series, not a split request, because a split would lose the dependence between assets.